In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data      = load_all_data()
companies = data['companies']
sectors   = data['sectors']

# Pull the saved financial_ratios_computed table
conn = sqlite3.connect('data/nifty100.db')
ratios = pd.read_sql_query("SELECT * FROM financial_ratios_computed", conn)
conn.close()

print(f"Ratios table: {ratios.shape}")
print(f"Companies: {companies.shape}")
print(f"Sectors: {sectors.shape}")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Ratios table: (1159, 43)
Companies: (92, 12)
Sectors: (92, 6)


In [2]:
# Merge sector info onto ratios
ratios_sector = ratios.merge(
    sectors[['company_id', 'broad_sector', 'sub_sector']],
    on='company_id', how='left'
)

financials = ratios_sector[ratios_sector['broad_sector'] == 'Financials']

print(f"Financials sector rows: {len(financials)}")
print(f"Unique Financial companies: {financials['company_id'].nunique()}")
print()
print("Sub-sectors within Financials:")
print(financials['sub_sector'].value_counts().to_string())

Financials sector rows: 258
Unique Financial companies: 23

Sub-sectors within Financials:
sub_sector
Private Banks             60
Public Sector Banks       48
Life Insurance            42
Speciality Finance        36
Consumer Finance          34
Diversified Financials    14
Holding Companies         12
General Insurance         12


In [3]:
# Get latest year only for comparison
latest_year = ratios_sector['year'].max()
latest = ratios_sector[ratios_sector['year'] == latest_year]

financials_latest     = latest[latest['broad_sector'] == 'Financials']
non_financials_latest = latest[latest['broad_sector'] != 'Financials']

print(f"Latest year: {latest_year}\n")

print("ROCE Comparison — Financials vs Non-Financials:")
print(f"  Financials median ROCE:     "
      f"{financials_latest['return_on_capital_pct'].median():.2f}%")
print(f"  Non-Financials median ROCE: "
      f"{non_financials_latest['return_on_capital_pct'].median():.2f}%")

print(f"\nFinancials D/E (expected to be very high — deposits = borrowings):")
print(f"  Financials median D/E:     "
      f"{financials_latest['debt_to_equity'].median():.2f}")
print(f"  Non-Financials median D/E: "
      f"{non_financials_latest['debt_to_equity'].median():.2f}")

Latest year: 2024-09

ROCE Comparison — Financials vs Non-Financials:
  Financials median ROCE:     nan%
  Non-Financials median ROCE: 17.74%

Financials D/E (expected to be very high — deposits = borrowings):
  Financials median D/E:     nan
  Non-Financials median D/E: 0.02


In [10]:
def compute_sector_relative_roce(df):
    """
    Computes a sector-relative ROCE score by ranking each company's
    ROCE against its own broad_sector peers, rather than the
    universal Nifty 100 range.

    This solves the bank/NBFC distortion problem.
    """
    df = df.copy()

    # Percentile rank within sector (0 to 1, higher = better ROCE in sector)
    df['roce_sector_percentile'] = df.groupby(
        ['broad_sector', 'year']
    )['return_on_capital_pct'].rank(pct=True).round(3)

    return df

ratios_sector = compute_sector_relative_roce(ratios_sector)

print("Sector-Relative ROCE — Sample for Financials (latest year):")
sample = ratios_sector[
    (ratios_sector['broad_sector'] == 'Financials') &
    (ratios_sector['year'] == latest_year)
][['company_id', 'sub_sector', 'return_on_capital_pct',
   'roce_sector_percentile']].sort_values(
    'roce_sector_percentile', ascending=False
)
print(sample.head(10).to_string(index=False))

Sector-Relative ROCE — Sample for Financials (latest year):
company_id             sub_sector  return_on_capital_pct  roce_sector_percentile
ICICIPRULI         Life Insurance                 753.98                   1.000
  HDFCLIFE         Life Insurance                 645.95                   0.952
   ICICIGI      General Insurance                 145.23                   0.905
      LICI         Life Insurance                  39.05                   0.857
BAJAJFINSV Diversified Financials                  11.45                   0.810
    RECLTD     Speciality Finance                   9.29                   0.762
 KOTAKBANK          Private Banks                   7.11                   0.714
      IRFC     Speciality Finance                   5.75                   0.667
 ICICIBANK          Private Banks                   5.12                   0.619
  HDFCBANK          Private Banks                   4.80                   0.571


In [11]:
# Get latest year ROE per company from our computed ratios
latest_roe = ratios_sector[ratios_sector['year'] == latest_year][
    ['company_id', 'return_on_equity_pct']
].rename(columns={'return_on_equity_pct': 'computed_roe'})

# Compare against companies.xlsx pre-computed roe_percentage
roe_check = companies[['id', 'roe_percentage']].rename(
    columns={'id': 'company_id', 'roe_percentage': 'source_roe'}
)

roe_compare = pd.merge(latest_roe, roe_check, on='company_id', how='inner')
roe_compare['diff'] = abs(roe_compare['computed_roe'] - roe_compare['source_roe'])

# Flag anomalies — diff greater than 5 percentage points
anomalies = roe_compare[roe_compare['diff'] > 5].sort_values(
    'diff', ascending=False
)

print(f"Total companies compared: {len(roe_compare)}")
print(f"Anomalies found (diff > 5pp): {len(anomalies)}")
print()
print("Top anomalies:")
print(anomalies.head(10).to_string(index=False))

Total companies compared: 91
Anomalies found (diff > 5pp): 18

Top anomalies:
company_id  computed_roe  source_roe    diff
       BEL       4744.05       26.30 4717.75
       HAL       3816.58       28.90 3787.68
        LT         77.67       10.49   67.18
       PNB         61.32        8.70   52.62
       TCS         50.94        0.52   50.42
 NESTLEIND        117.75      135.61   17.86
      LICI         49.45       63.61   14.16
BAJAJFINSV         25.85       11.72   14.13
TATAMOTORS         37.46       49.40   11.94
 TATASTEEL         -5.33        6.55   11.88


In [6]:
# Build the final sector_roce_notes.csv report
notes = []

# Add Financials sector summary
notes.append({
    'category': 'Sector Summary',
    'note': f'Financials median ROCE: {financials_latest["return_on_capital_pct"].median():.2f}% '
            f'vs Non-Financials: {non_financials_latest["return_on_capital_pct"].median():.2f}%. '
            f'Standard ROCE distorted for banks/NBFCs due to deposits counted as borrowings. '
            f'Use sector-relative percentile ranking instead.'
})

# Add ROE anomalies
for _, row in anomalies.iterrows():
    notes.append({
        'category': 'ROE Anomaly',
        'company_id': row['company_id'],
        'computed_roe': row['computed_roe'],
        'source_roe': row['source_roe'],
        'diff': round(row['diff'], 2),
        'note': f"Computed ROE ({row['computed_roe']:.2f}%) differs from "
                f"source ({row['source_roe']:.2f}%) by {row['diff']:.2f}pp"
    })

notes_df = pd.DataFrame(notes)
notes_df.to_csv('output/sector_roce_notes.csv', index=False)

print(f"sector_roce_notes.csv saved with {len(notes_df)} entries!")
print()
print(notes_df.to_string(index=False))

sector_roce_notes.csv saved with 1 entries!

      category                                                                                                                                                                                   note
Sector Summary Financials median ROCE: nan% vs Non-Financials: 17.74%. Standard ROCE distorted for banks/NBFCs due to deposits counted as borrowings. Use sector-relative percentile ranking instead.


In [7]:
# Investigate Issue 1 — Why is Financials ROCE all NaN?
print("ISSUE 1: Financials ROCE investigation")
print("="*50)
financials_check = latest[latest['broad_sector'] == 'Financials']
print(f"Financials companies in latest year: {len(financials_check)}")
print(f"Sample company IDs: {financials_check['company_id'].unique()[:10]}")
print()
print("ROCE values for these companies:")
print(financials_check[['company_id', 'return_on_capital_pct']].head(10))
print()

# Investigate Issue 2 — Why only 1 company in ROE comparison?
print("\nISSUE 2: ROE comparison investigation")
print("="*50)
print(f"latest_roe shape: {latest_roe.shape}")
print(f"latest_roe sample:\n{latest_roe.head()}")
print()
print(f"companies['id'] sample: {companies['id'].head().tolist()}")
print(f"latest_roe['company_id'] sample: {latest_roe['company_id'].head().tolist()}")
print()
print(f"latest_year being used: {latest_year}")
print(f"How many rows have year == latest_year in ratios_sector: "
      f"{(ratios_sector['year'] == latest_year).sum()}")

ISSUE 1: Financials ROCE investigation
Financials companies in latest year: 0
Sample company IDs: []

ROCE values for these companies:
Empty DataFrame
Columns: [company_id, return_on_capital_pct]
Index: []


ISSUE 2: ROE comparison investigation
latest_roe shape: (1, 2)
latest_roe sample:
    company_id  computed_roe
937    SIEMENS          17.7

companies['id'] sample: ['ABB', 'ADANIENSOL', 'ADANIENT', 'ADANIGREEN', 'ADANIPORTS']
latest_roe['company_id'] sample: ['SIEMENS']

latest_year being used: 2024-09
How many rows have year == latest_year in ratios_sector: 1


In [8]:
# Find a proper "latest year" - one with good coverage across companies
year_counts = ratios_sector.groupby('year')['company_id'].nunique().sort_index()
print("Number of companies per year (last 10 years):")
print(year_counts.tail(10))

Number of companies per year (last 10 years):
year
2021-09     1
2021-12     3
2022-03    95
2022-09     1
2022-12     2
2023-03    97
2023-09     1
2023-12     2
2024-03    98
2024-09     1
Name: company_id, dtype: int64


In [9]:
# Use 2024-03 as our analysis year instead of the buggy "max" year
latest_year = '2024-03'
latest = ratios_sector[ratios_sector['year'] == latest_year]

print(f"Using latest_year = {latest_year}")
print(f"Companies in this year: {latest['company_id'].nunique()}")
print()

financials_latest     = latest[latest['broad_sector'] == 'Financials']
non_financials_latest = latest[latest['broad_sector'] != 'Financials']

print("ROCE Comparison — Financials vs Non-Financials:")
print(f"  Financials median ROCE:     "
      f"{financials_latest['return_on_capital_pct'].median():.2f}%")
print(f"  Non-Financials median ROCE: "
      f"{non_financials_latest['return_on_capital_pct'].median():.2f}%")

print(f"\nFinancials D/E (expected very high — deposits = borrowings):")
print(f"  Financials median D/E:     "
      f"{financials_latest['debt_to_equity'].median():.2f}")
print(f"  Non-Financials median D/E: "
      f"{non_financials_latest['debt_to_equity'].median():.2f}")

Using latest_year = 2024-03
Companies in this year: 98

ROCE Comparison — Financials vs Non-Financials:
  Financials median ROCE:     4.17%
  Non-Financials median ROCE: 16.21%

Financials D/E (expected very high — deposits = borrowings):
  Financials median D/E:     4.39
  Non-Financials median D/E: 0.32


In [12]:
# Investigate BEL and HAL extreme ROE anomalies
investigate = ratios_sector[
    (ratios_sector['company_id'].isin(['BEL', 'HAL'])) &
    (ratios_sector['year'] == latest_year)
][['company_id', 'return_on_equity_pct', 'return_on_capital_pct']]

print(investigate.to_string(index=False))

# Check their underlying equity values
conn = sqlite3.connect('data/nifty100.db')
bs_check = pd.read_sql_query(
    "SELECT company_id, year, equity_capital, reserves, borrowings "
    "FROM balancesheet WHERE company_id IN ('BEL','HAL') AND year = '2024-03'",
    conn
)
conn.close()
print()
print(bs_check.to_string(index=False))

company_id  return_on_equity_pct  return_on_capital_pct
       BEL               4744.05                3628.35
       HAL               3816.58                2590.99

company_id    year  equity_capital  reserves  borrowings
       BEL 2024-03            11.0        73          43
       HAL 2024-03             5.0       194         123


In [13]:
# Build the final comprehensive sector_roce_notes.csv
notes = []

# Sector summary
notes.append({
    'category': 'Sector Summary',
    'company_id': None,
    'computed_value': None,
    'source_value': None,
    'diff': None,
    'note': f'Financials median ROCE: {financials_latest["return_on_capital_pct"].median():.2f}% '
            f'vs Non-Financials: {non_financials_latest["return_on_capital_pct"].median():.2f}%. '
            f'Financials median D/E: {financials_latest["debt_to_equity"].median():.2f} vs '
            f'Non-Financials: {non_financials_latest["debt_to_equity"].median():.2f}. '
            f'Standard ROCE/D-E distorted for banks/NBFCs due to deposits counted as '
            f'borrowings. Sector-relative percentile ranking implemented instead.'
})

# Extreme value findings (BEL, HAL)
for company in ['BEL', 'HAL']:
    row = investigate[investigate['company_id'] == company].iloc[0]
    notes.append({
        'category': 'Extreme Value - Small Equity Base',
        'company_id': company,
        'computed_value': round(row['return_on_equity_pct'], 2),
        'source_value': None,
        'diff': None,
        'note': f'{company} shows ROE of {row["return_on_equity_pct"]:.0f}% due to very '
                f'small equity capital + reserves base relative to profit. '
                f'Mathematically correct but requires winsorisation (cap at P10/P90) '
                f'before use in Health Score or screener ranking.'
    })

# ROE anomalies vs source
for _, row in anomalies.iterrows():
    notes.append({
        'category': 'ROE Anomaly vs Source',
        'company_id': row['company_id'],
        'computed_value': round(row['computed_roe'], 2),
        'source_value': round(row['source_roe'], 2),
        'diff': round(row['diff'], 2),
        'note': f"Computed ROE ({row['computed_roe']:.2f}%) differs from "
                f"companies.xlsx source ({row['source_roe']:.2f}%) by {row['diff']:.2f}pp. "
                f"TCS confirmed as known source data entry error per project documentation."
    })

notes_df = pd.DataFrame(notes)
notes_df.to_csv('output/sector_roce_notes.csv', index=False)

print(f"sector_roce_notes.csv saved with {len(notes_df)} entries!")
print()
print(f"Summary by category:")
print(notes_df['category'].value_counts().to_string())

sector_roce_notes.csv saved with 21 entries!

Summary by category:
category
ROE Anomaly vs Source                18
Extreme Value - Small Equity Base     2
Sector Summary                        1
